In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import glob
import time
import numpy as np
import pandas as pd
from scipy.sparse import diags, kron, eye, csr_matrix, bmat, random as sp_random
from scipy.sparse.linalg import gmres, splu
import warnings

warnings.filterwarnings("ignore")
from IPython.display import HTML, Javascript
HTML('''<script>
function ClickConnect(){console.log("Keep alive"); document.querySelector("colab-connect-button").click()}
setInterval(ClickConnect, 60000);
</script>''')\
HELMHOLTZ_SINGULAR_PROB = 0.30
WAVE_EXTREME_PROB       = 0.22

# ======================  HELPER FUNCTIONS ======================

def build_1d_base(n, bc="dirichlet", rng=None):
    if rng is None:
        rng = np.random.default_rng()
    main = 2.0 * np.ones(n)
    off  = -1.0 * np.ones(n - 1)
    T    = diags([main, off, off], [0, -1, 1], format="lil")
    if bc == "neumann":
        T[0, 0] = T[n-1, n-1] = 1.0
    elif bc == "robin":
        alpha = rng.uniform(0.5, 5.0)
        T[0, 0] = T[n-1, n-1] = 2.0 + alpha
    return csr_matrix(T)

def apply_symmetric_stretching(A, N, rng):
    s = np.exp(rng.uniform(-1.4, 1.4, size=N))
    sqrtD = diags(np.sqrt(s), format="csr")
    return sqrtD @ A @ sqrtD

def apply_grid_stretching(A, N, rng):
    s = np.exp(rng.uniform(-1.4, 1.4, size=N))
    return diags(s, format="csr") @ A

def apply_sparsity_noise(A, N, rng, density=0.0015, symmetric=False):
    fro       = np.sqrt(np.sum(A.data**2) + 1e-12)
    noise_mag = 5e-4 * fro / (N + 1e-12)
    E         = sp_random(N, N, density=density, format="csr",
                          random_state=int(rng.integers(0, 2**31-1)))
    E.data   *= noise_mag
    if symmetric:
        E = (E + E.T) * 0.5
    return A + E

def robust_smax(A, rng, n_iter=25):
    try:
        N = A.shape[0]
        x = rng.standard_normal(N)
        x /= np.linalg.norm(x) + 1e-12
        for _ in range(n_iter):
            x   = A.T @ (A @ x)
            nrm = np.linalg.norm(x)
            if nrm < 1e-12: break
            x  /= nrm
        smax = np.sqrt(max(np.dot(x, A.T @ (A @ x)), 0.0))
        return max(smax, 1e-6)
    except:
        return np.sqrt(np.sum(A.data**2) + 1e-12)

def robust_smin(A, reg_shift=0.0, n_iter=15):
    try:
        N = A.shape[0]
        if not np.isfinite(A.data).all():
            return None
        fro = np.sqrt(np.sum(A.data**2))
        if fro > 1e30 or fro < 1e-30:
            scale       = fro
            A_scaled    = A * (1.0 / scale)
            smin_scaled = robust_smin(A_scaled, reg_shift / scale, n_iter)
            return smin_scaled * scale if smin_scaled is not None else None
        if reg_shift > 0:
            M = A.T @ A + (reg_shift**2) * eye(N, format="csr")
        else:
            M = A.T @ A
        lu = splu(M.tocsc(), permc_spec='COLAMD')
        x  = np.ones(N, dtype=np.float64) / np.sqrt(N)
        for _ in range(n_iter):
            x   = lu.solve(x)
            nrm = np.linalg.norm(x)
            if nrm < 1e-12: return None
            x  /= nrm
        rq        = np.dot(x, M @ x)
        shift_sq  = reg_shift ** 2 if reg_shift > 0 else 0.0
        eigenval  = max(rq - shift_sq, 1e-30)
        smin      = np.sqrt(eigenval)
        return max(smin, 1e-15)
    except:
        return None

def cond_estimate_lu(A, already_normalized=False, rng=None):
    try:
        N = A.shape[0]
        if not np.isfinite(A.data).all():
            return None
        if not already_normalized:
            fro = np.sqrt(np.sum(A.data**2))
            if fro < 1e-30 or fro > 1e30:
                return None
            A_scaled = A * (1.0 / fro)
        else:
            A_scaled = A
        norm_A   = float(np.max(np.abs(A_scaled).sum(axis=0)))
        lu       = splu(A_scaled.tocsc(), permc_spec='COLAMD')
        rng_p    = rng if rng is not None else np.random.default_rng()
        est      = 0.0
        for _ in range(6):
            e   = rng_p.choice([-1.0, 1.0], size=N)
            x   = lu.solve(e)
            est = max(est, np.linalg.norm(x, 1) / (np.linalg.norm(e, 1) + 1e-12))
        cond   = norm_A * est
        return float(np.log10(max(cond, 1.0) + 1e-12))
    except:
        return None

def gmres_decay_feature(A, N):
    b         = np.ones(N, dtype=np.float64)
    residuals = []
    try:
        def cb(r):
            residuals.append(float(np.linalg.norm(r)) if hasattr(r, '__len__') else float(r))
        gmres(A, b, maxiter=8, callback=cb, callback_type='pr_norm', atol=1e-8)
    except:
        pass
    if len(residuals) > 1:
        return float(np.clip(residuals[-1] / (residuals[0] + 1e-12), 0.0, 10.0))
    return 1.0

# ====================== FEATURE HELPERS ======================
def bandwidth_estimate(A):
    rows, cols = A.nonzero()
    return float(np.max(np.abs(rows - cols))) if len(rows) > 0 else 0.0

def sparsity_entropy(A):
    nnz_per_row = A.getnnz(axis=1)
    p = nnz_per_row.astype(float) / (nnz_per_row.sum() + 1e-12)
    p = p[p > 0]
    return -np.sum(p * np.log2(p + 1e-12)) if len(p) > 0 else 0.0

def off_diag_decay(A, rng, n_samples=100):
    N = A.shape[0]
    indptr, indices, data = A.indptr, A.indices, A.data
    rates = []
    n_draw = min(n_samples, N)
    idxs = rng.integers(0, N, size=n_draw)
    for i in idxs:
        start, end = indptr[i], indptr[i + 1]
        nnz = end - start
        if nnz > 3:
            row_idx = indices[start:end]
            row_val = data[start:end]
            dists      = np.abs(row_idx - i)
            vals       = np.abs(row_val)
            sorted_idx = dists.argsort()
            rates.append(np.mean(vals[sorted_idx][1:5]) /
                         (vals[sorted_idx[0]] + 1e-12))
    raw = np.mean(rates) if rates else 1.0
    return float(np.log1p(raw))

def gershgorin_features(diag, row_abs_sums):
    radii = row_abs_sums - np.abs(diag)
    return {
        "gersh_radius_max":         float(radii.max()),
        "gersh_radius_mean":        float(radii.mean()),
        "gersh_radius_std":         float(radii.std()),
        "gersh_origin_overlap_pct": float(np.mean(radii >= np.abs(diag))),
        "gersh_min_gap":            float(np.min(np.abs(diag) - radii)),
    }

def spectral_features(A, rng, diag, row_abs_sums, k=3):
    N      = A.shape[0]
    result = {}
    try:
        Q      = rng.standard_normal((N, k + 1))
        Q, _   = np.linalg.qr(Q)
        M_small = Q.T @ (A.T @ (A @ Q))
        eigs   = np.sort(np.abs(np.linalg.eigvalsh(M_small)))[::-1]
        for i, e in enumerate(eigs[:k]):
            result[f"rayleigh_eig{i}"] = float(e)
    except:
        for i in range(k):
            result[f"rayleigh_eig{i}"] = 0.0
    try:
        diag_abs    = np.abs(diag)
        row_offdiag = row_abs_sums - diag_abs
        dominance   = diag_abs - row_offdiag
        result["dom_q10"] = float(np.percentile(dominance, 10))
        result["dom_q50"] = float(np.percentile(dominance, 50))
        result["dom_q90"] = float(np.percentile(dominance, 90))
    except:
        result["dom_q10"] = result["dom_q50"] = result["dom_q90"] = 0.0
    return result

def row_col_statistics(A, diag, row_abs_sums):
    row_sums     = np.array(A.sum(axis=1)).flatten()
    col_sums     = np.array(A.sum(axis=0)).flatten()
    off_diag_max = row_abs_sums - np.abs(diag)
    return {
        "row_sum_min":         float(row_sums.min()),
        "row_sum_max":         float(row_sums.max()),
        "row_sum_std":         float(row_sums.std()),
        "col_sum_min":         float(col_sums.min()),
        "col_sum_max":         float(col_sums.max()),
        "col_sum_std":         float(col_sums.std()),
        "min_diag_dominance":  float(np.min(np.abs(diag) - off_diag_max)),
        "avg_diag_dominance":  float(np.mean(np.abs(diag) - off_diag_max)),
        "diag_dominant_ratio": float(np.mean(np.abs(diag) > off_diag_max)),
    }
# ====================== PDE BUILDERS ======================

def build_poisson(n, N, h, bc, rng):
    I    = eye(n, format="csr")
    Tx   = build_1d_base(n, bc, rng)
    Ty   = build_1d_base(n, bc, rng)
    cx   = 10 ** rng.uniform(-4.5, 6.0)
    cy   = 10 ** rng.uniform(-4.5, 6.0)
    jump = 10 ** rng.uniform(0.4, 8.0)
    L    = (cx * kron(I, Tx) + cy * kron(Ty, I)) / h**2
    coeff = np.ones(N); coeff[N//2:] *= jump
    sqrtD = diags(np.sqrt(coeff), format="csr")
    return csr_matrix(sqrtD @ L @ sqrtD, dtype=np.float64), 0.0, "poisson"

def build_convdiff(n, N, h, bc, rng):
    I    = eye(n, format="csr")
    Tx   = build_1d_base(n, bc, rng)
    Ty   = build_1d_base(n, bc, rng)
    ex   = 10 ** rng.uniform(-12.0, -1.5)
    ey   = 10 ** rng.uniform(-12.0, -1.5)
    v    = 10 ** rng.uniform(-1.0, 4.0)
    jump = 10 ** rng.uniform(2.0, 9.0)
    L    = (ex * kron(I, Tx) + ey * kron(Ty, I)) / h**2
    coeff = np.ones(N); coeff[N//2:] *= jump
    D    = diags(coeff, format="csr")
    e = np.ones(N)
    Adv = (v / h) * diags([e, -e[:-1]], [0, -1], shape=(N, N), format="csr")
    return csr_matrix(D @ L + D @ Adv, dtype=np.float64), 0.0, "convection_diffusion"

def build_helmholtz(n, N, h, bc, rng):

    I   = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L   = (kron(I, T) + kron(T, I)) / h**2

    eigs_1d = np.sort(np.linalg.eigvalsh(T.toarray())) / h**2
    eigs_1d = np.clip(eigs_1d, 0.0, None)

    lam_min = 2.0 * eigs_1d[0]
    lam_max = 2.0 * eigs_1d[-1]

    i_idx = int(rng.integers(0, n))
    j_idx = int(rng.integers(0, n))
    lam_target = eigs_1d[i_idx] + eigs_1d[j_idx]
    lam_target = max(lam_target, 1e-10)

    if rng.random() < HELMHOLTZ_SINGULAR_PROB:
        target_log10 = rng.uniform(5.5, 13.0)
        gap = lam_target / (10 ** target_log10)
        sign = rng.choice([-1.0, 1.0])
        k_sq = lam_target - sign * gap * rng.uniform(0.9, 1.3)
    else:
        if rng.random() < 0.65:
            k_sq = -lam_max * rng.uniform(3.0, 30.0)
        else:
            k_sq = lam_min * rng.uniform(0.4, 1.2)

    k_sq = float(np.clip(k_sq, -400.0 * lam_max, lam_max * 0.9995))

    if abs(lam_target - k_sq) < 1e-12 * max(lam_target, 1.0):
        k_sq -= max(lam_target, 1.0) * 3e-8

    A = csr_matrix(L - k_sq * eye(N, format="csr"), dtype=np.float64)
    return A, 0.0, "helmholtz"

def build_advreact(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)
    if rng.random() < 0.70:
        eps   = 10 ** rng.uniform(-12.0, -4.0)
        sigma = 10 ** rng.uniform(5.0, 11.0)
        jump  = 10 ** rng.uniform(5.0, 14.0)
    else:
        eps   = 10 ** rng.uniform(-9.0, -2.0)
        sigma = 10 ** rng.uniform(-3.5, 4.5)
        jump  = 10 ** rng.uniform(3.0, 12.0)
    L     = (eps * kron(I, Tx) + eps * kron(Ty, I)) / h**2
    coeff = np.full(N, sigma); coeff[N//2:] = sigma / jump
    return csr_matrix(L + diags(coeff, format="csr"),
                      dtype=np.float64), 0.0, "advection_reaction"

def build_biharmonic(n, N, h, bc, rng):
    I     = eye(n, format="csr")
    T     = build_1d_base(n, bc, rng)
    L     = (kron(I, T) + kron(T, I)) / h**2
    scale = 10 ** rng.uniform(0.0, 3.5)
    sqrtD = diags(np.sqrt(scale) * np.ones(N), format="csr")
    Ls    = sqrtD @ L @ sqrtD
    return csr_matrix(Ls @ Ls, dtype=np.float64), 0.0, "biharmonic"

def build_stokes_saddle(n, N, h, bc, rng):

    I_n = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L2d = (kron(I_n, T) + kron(T, I_n)) / h**2

    e  = np.ones(N)
    Bx = (1.0 / h) * diags([-e, e[:-1]], [0, 1], shape=(N, N), format="csr")
    By = (1.0 / h) * diags([e, -e[:N - n]], [0, n],  shape=(N, N), format="csr")

    B_scale = 1.0 / h

    if rng.random() < 0.45:
        nu    = 10 ** rng.uniform(0.5, 2.5)
        eps_p = 10 ** rng.uniform(-3.0, 0.0)
    else:
        nu    = 10 ** rng.uniform(-5.0, -2.0)
        eps_p = 10 ** rng.uniform(-12.0, -6.0)

    A = bmat([[nu * L2d,           csr_matrix((N, N)), Bx.T],
              [csr_matrix((N, N)), nu * L2d,           By.T],
              [Bx,                 By, -eps_p * eye(N, format="csr")]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "stokes_saddle"


def build_tensor_diffusion(n, N, h, bc, rng):
    I      = eye(n, format="csr")
    T      = build_1d_base(n, bc, rng)
    max_log = rng.uniform(3.5, 9.0)
    log_a1  = np.linspace(0, max_log, N);                  rng.shuffle(log_a1)
    log_a2  = np.linspace(0, max_log * rng.uniform(0.4, 0.9), N); rng.shuffle(log_a2)
    Kx = diags(10**log_a1, format="csr")
    Ky = diags(10**log_a2, format="csr")
    A  = (Kx @ kron(I, T) + Ky @ kron(T, I)) / h**2
    if rng.random() < 0.5:
        theta = rng.uniform(0, np.pi / 3)
        cross = 10 ** rng.uniform(-2.5, 1.0)
        off   = diags([np.cos(theta) * np.sin(theta) * cross * np.ones(N - 1)] * 2,
                      [-1, 1], format="csr")
        A = A + off
    if rng.random() < 0.45:
        A = A * 10 ** rng.uniform(-1.5, 3.0)
    return csr_matrix(A, dtype=np.float64), 0.0, "tensor_diffusion"

def build_parabolic(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    T  = build_1d_base(n, bc, rng)
    K  = (kron(I, T) + kron(T, I)) / h**2
    dt = 10 ** rng.uniform(-8.0, 5.0)
    M  = diags(10 ** rng.uniform(-8.0, 8.0, size=N), format="csr")
    A  = M + dt * K
    if rng.random() < 0.75: A = A * 10 ** rng.uniform(-4.0, 8.0)
    if rng.random() < 0.55: A = A * 10 ** rng.uniform(4.0, 10.0)
    return csr_matrix(A, dtype=np.float64), 0.0, "parabolic"

def build_hyperbolic(n, N, h, bc, rng):

    vx  = 10 ** rng.uniform(-1.0, 7.0) * rng.choice([-1, 1])
    vy  = 10 ** rng.uniform(-1.0, 7.0) * rng.choice([-1, 1])
    eps = 10 ** rng.uniform(-8.0, 1.5)

    e = np.ones(N)
    if vx >= 0:
        Adv_x = (abs(vx) / h) * diags([e, -e[:-1]],  [0, -1], shape=(N, N), format="csr")
    else:
        Adv_x = (abs(vx) / h) * diags([-e, e[:-1]], [0,  1], shape=(N, N), format="csr")
    if vy >= 0:
        Adv_y = (abs(vy) / h) * diags([np.ones(N), -np.ones(N - n)],
                                       [0, -n], shape=(N, N), format="csr")
    else:
        Adv_y = (abs(vy) / h) * diags([-np.ones(N), np.ones(N - n)],
                                       [0,  n], shape=(N, N), format="csr")

    if rng.random() < 0.50:
        log_scale = rng.uniform(0.0, 10.0)
        d_vals    = np.ones(N)
        idx       = rng.choice(N, size=N // 2, replace=False)
        d_vals[idx] = 10 ** log_scale
        D   = diags(d_vals, format="csr")
        A   = D @ (Adv_x + Adv_y) + eps * D
    else:
        A = Adv_x + Adv_y + eps * eye(N, format="csr")

    return csr_matrix(A, dtype=np.float64), 0.0, "hyperbolic"

def build_anisotropic(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)

    if rng.random() < 0.50:
        log_base  = rng.uniform(-1.0, 1.0)
        log_ratio = rng.uniform(-1.5, 1.5)
    else:
        log_base  = rng.uniform(-1.0, 1.0)
        sign      = rng.choice([-1.0, 1.0])
        log_ratio = sign * rng.uniform(8.0, 14.0)

    ax = 10 ** log_base
    ay = 10 ** (log_base + log_ratio)
    ay = float(np.clip(ay, 1e-15, 1e15))

    return csr_matrix((ax * kron(I, Tx) + ay * kron(Ty, I)) / h**2,
                      dtype=np.float64), 0.0, "anisotropic"

def build_mixed(n, N, h, bc, rng):

    if rng.random() < 0.48:

        I    = eye(n, format="csr")
        Tx   = build_1d_base(n, bc, rng)
        Ty   = build_1d_base(n, bc, rng)
        c    = 10 ** rng.uniform(-1.0, 2.0)
        L    = (c * kron(I, Tx) + c * kron(Ty, I)) / h**2
        A1   = csr_matrix(L, dtype=np.float64)
        A2   = csr_matrix(L * (10 ** rng.uniform(-0.5, 0.5)), dtype=np.float64)
        coupling = 10 ** rng.uniform(-8.0, -4.0)
    else:
        A1, _, _ = build_poisson(n, N, h, bc, rng)
        A2, _, _ = build_advreact(n, N, h, bc, rng)
        coupling = 10 ** rng.uniform(-1.0, 4.0)

    A = bmat([[A1, coupling * eye(N, format="csr")],
              [coupling * eye(N, format="csr"), A2]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "mixed"

def build_reaction_diffusion(n, N, h, bc, rng):
    I_n = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L   = (kron(I_n, T) + kron(T, I_n)) / h**2
    if rng.random() < 0.82:
        a     = 10 ** rng.uniform(4.0, 10.0)
        delta = 10 ** rng.uniform(-8.0, -3.0)
        b     = a * (1.0 - delta)
        eps1  = 10 ** rng.uniform(-10.0, -4.0)
        eps2  = 10 ** rng.uniform(-10.0, -4.0)
        A11   =  eps1 * L - a * eye(N, format="csr")
        A22   =  eps2 * L - a * eye(N, format="csr")
        A12   = -b * eye(N, format="csr")
        A21   = -b * eye(N, format="csr")
    else:
        d1  = 10 ** rng.uniform(-3.0, 2.0)
        d2  = 10 ** rng.uniform(-3.0, 2.0)
        a11 = 10 ** rng.uniform(-2.0, 3.0) * rng.choice([-1, 1])
        a12 = 10 ** rng.uniform(-2.0, 2.0)
        a21 = 10 ** rng.uniform(-2.0, 2.0)
        a22 = 10 ** rng.uniform(-2.0, 3.0) * rng.choice([-1, 1])
        A11 = d1 * L - a11 * eye(N, format="csr")
        A12 = -a12 * eye(N, format="csr")
        A21 = -a21 * eye(N, format="csr")
        A22 = d2 * L - a22 * eye(N, format="csr")
    A = bmat([[A11, A12], [A21, A22]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "reaction_diffusion"


def build_wave_implicit(n, N, h, bc, rng):

    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)

    m0 = 10 ** rng.uniform(-1.0, 1.0)

    lam_max_L = (2.0 / h**2) * (1.0 - np.cos(np.pi * n / (n + 1)))

    extreme = rng.random() < WAVE_EXTREME_PROB

    if rng.random() < 0.50:
        rho = 10 ** rng.uniform(7.0, 11.0) if extreme else 10 ** rng.uniform(-3.0, 1.5)
        cx         = 10 ** rng.uniform(-1.0, 2.0)
        cy         = cx * 10 ** rng.uniform(-0.5, 0.5)
        dt_sq      = rho * m0 / (cx * lam_max_L + 1e-30)
        dt         = float(np.sqrt(np.clip(dt_sq, 0, 1e20)))
    else:
        cx              = 10 ** rng.uniform(1.0, 4.0)
        aniso_ratio     = 10 ** rng.uniform(5.0, 9.0)
        cy              = cx / aniso_ratio
        if rng.random() < 0.5:
            cx, cy = cy, cx
        rho = 10 ** rng.uniform(7.0, 11.0) if extreme else 10 ** rng.uniform(0.0, 4.0)
        dt_sq = rho * m0 / (max(cx, cy) * lam_max_L + 1e-30)
        dt    = float(np.sqrt(np.clip(dt_sq, 0, 1e20)))

    K = (cx * kron(I, Tx) + cy * kron(Ty, I)) / h**2
    M = m0 * eye(N, format="csr")
    A = M + dt**2 * K

    A = csr_matrix(A, dtype=np.float64)
    if not np.isfinite(A.data).all():
        A.data = np.clip(A.data, -1e30, 1e30)

    return A, 0.0, "wave_implicit"

def build_heterogeneous_diffusion(n, N, h, bc, rng):
    I = eye(n, format="csr")
    T = build_1d_base(n, bc, rng)
    L = (kron(I, T) + kron(T, I)) / h**2
    k = np.ones(N)
    k[N//4:N//2] = 10 ** rng.uniform(0.5, 8.0)
    k[N//2:]     = 10 ** rng.uniform(0.8, 9.0)
    if rng.random() < 0.5: k *= 10 ** rng.uniform(-1.8, 1.8)
    sqrtD = diags(np.sqrt(k), format="csr")
    return csr_matrix(sqrtD @ L @ sqrtD, dtype=np.float64), 0.0, "heterogeneous_diffusion"

# ====================== BUILDERS DICT ======================
BUILDERS = {
    "poisson":                  build_poisson,
    "convection_diffusion":     build_convdiff,
    "helmholtz":                build_helmholtz,
    "advection_reaction":       build_advreact,
    "biharmonic":               build_biharmonic,
    "stokes_saddle":            build_stokes_saddle,
    "tensor_diffusion":         build_tensor_diffusion,
    "parabolic":                build_parabolic,
    "hyperbolic":               build_hyperbolic,
    "anisotropic":              build_anisotropic,
    "mixed":                    build_mixed,
    "reaction_diffusion":       build_reaction_diffusion,
    "wave_implicit":            build_wave_implicit,
    "heterogeneous_diffusion":  build_heterogeneous_diffusion,
}

# ====================== MAKE SAMPLE ======================
def make_sample(pde, grid_size, bc, rng):
    n      = int(grid_size)
    base_N = n * n
    h      = 1.0 / (n + 1)

    A, reg_shift, pde_name = BUILDERS[pde](n, base_N, h, bc, rng)
    N = A.shape[0]

    spd_families = {"poisson", "biharmonic", "parabolic",
                    "heterogeneous_diffusion", "tensor_diffusion"}
    is_spd = pde_name in spd_families

    if is_spd and rng.random() < 0.6:
        A = apply_symmetric_stretching(A, N, rng)
    elif rng.random() < 0.4:
        A = apply_grid_stretching(A, N, rng)

    if rng.random() < 0.25:
        A = apply_sparsity_noise(A, N, rng, symmetric=is_spd)

    fro = np.sqrt(np.sum(A.data ** 2))
    if fro < 1e-30 or not np.isfinite(fro):
        return None
    A_norm = A * (1.0 / fro)

    log10c = cond_estimate_lu(A_norm, already_normalized=True, rng=rng)

    if log10c is None:
        try:
            smax = robust_smax(A_norm, rng)
            smin = robust_smin(A_norm, reg_shift / fro)
            if smin is not None and smin > 1e-14:
                cond_num = smax / smin
                log10c   = float(np.log10(cond_num + 1e-12))
            else:
                return None
        except:
            return None

    cond_num = 10 ** log10c
    diag     = A_norm.diagonal()
    fro_norm = np.sqrt(np.sum(A_norm.data**2) + 1e-12)

    all_pde_names = list(BUILDERS.keys())
    abs_A_norm    = A_norm.copy()
    abs_A_norm.data = np.abs(abs_A_norm.data)
    row_abs_sums  = np.array(abs_A_norm.sum(axis=1)).flatten()
    col_abs_sums  = np.array(abs_A_norm.sum(axis=0)).flatten()
    matrix_1norm   = float(col_abs_sums.max())
    matrix_infnorm = float(row_abs_sums.max())
    diff = A_norm - A_norm.T
    symmetry_measure = float(np.sqrt(diff.multiply(diff).sum()) / fro_norm)

    features = {
        "diag_min":         float(diag.min()),
        "diag_max":         float(diag.max()),
        "diag_mean":        float(diag.mean()),
        "diag_std":         float(diag.std()),
        "diag_ratio":       float(np.min(np.abs(diag)) / (np.max(np.abs(diag)) + 1e-12)),
        "frobenius_norm":   float(fro_norm),
        "matrix_1norm":     matrix_1norm,
        "matrix_infnorm":   matrix_infnorm,
        "norm_ratio_1_inf": matrix_1norm / (matrix_infnorm + 1e-12),
        "symmetry_measure": symmetry_measure,
        "avg_nnz_row":      A_norm.nnz / N,
        "max_row_nnz":      int(A_norm.getnnz(axis=1).max()),
        "nnz_std":          float(np.std(A_norm.getnnz(axis=1))),
        "bandwidth":        bandwidth_estimate(A_norm),
        "bandwidth_ratio":  bandwidth_estimate(A_norm) / np.sqrt(N),
        "sparsity_entropy": sparsity_entropy(A_norm),
        "off_diag_decay":   off_diag_decay(A_norm, rng),
        **gershgorin_features(diag, row_abs_sums),
        **row_col_statistics(A_norm, diag, row_abs_sums),
        **spectral_features(A_norm, rng, diag, row_abs_sums),
        "gmres_decay":      gmres_decay_feature(A_norm, N),
        "mesh_size":        float(h),
        "grid_size":        int(grid_size),
        "block_size":       3 if pde_name == "stokes_saddle" else (
                            2 if pde_name in {"mixed", "reaction_diffusion"} else 1),
        "pde_type":         pde_name,
        "condition_number": float(cond_num),
        "log10_cond":       log10c,
        "ill_conditioned":  int(log10c >= 7.0),
    }
    for p in all_pde_names:
        features[f"is_{p}"] = int(pde == p)
    for b in ["dirichlet", "neumann", "robin"]:
        features[f"bc_{b}"] = int(bc == b)

    return features


def generate_dataset_chunked(n_samples=8000, batch_size=300,
                             out_dir="matrix_batches",
                             final_path="matrix_dataset_z.csv",
                             seed=42):
    os.makedirs(out_dir, exist_ok=True)

    grid_sizes = [20, 25, 30, 35, 40, 45, 50, 80]
    all_pdes = list(BUILDERS.keys())
    weights = np.ones(len(all_pdes), dtype=float)
    weights /= weights.sum()

    n_batches = (n_samples + batch_size - 1) // batch_size

    existing = sorted(glob.glob(os.path.join(out_dir, "batch_*.csv")))
    done_batches = {int(os.path.basename(f).replace("batch_", "").replace(".csv", ""))
                    for f in existing}

    print(f"Found {len(done_batches)} completed batches. Resuming...")

    t0 = time.time()
    for b in range(n_batches):
        if b in done_batches:
            continue

        batch_path = os.path.join(out_dir, f"batch_{b:04d}.csv")
        target_n = min(batch_size, n_samples - b * batch_size)

        batch_rng = np.random.default_rng(seed + b)

        dataset = []
        skipped = 0
        i = 0
        while i < target_n:
            pde = batch_rng.choice(all_pdes, p=weights)
            gs = batch_rng.choice(grid_sizes)
            bc = batch_rng.choice(["dirichlet", "neumann", "robin"])

            sample = make_sample(pde, gs, bc, batch_rng)
            if sample:
                dataset.append(sample)
                i += 1
            else:
                skipped += 1
                for _ in range(3):
                    sample = make_sample(batch_rng.choice(all_pdes, p=weights),
                                         batch_rng.choice(grid_sizes),
                                         batch_rng.choice(["dirichlet", "neumann", "robin"]),
                                         batch_rng)
                    if sample:
                        dataset.append(sample)
                        i += 1
                        break

        batch_df = pd.DataFrame(dataset)
        batch_df.to_csv(batch_path, index=False)

        elapsed = (time.time() - t0) / 60
        print(f"Batch {b+1:3d}/{n_batches} | size={len(batch_df)} | skipped={skipped} | "
              f"time={elapsed:.1f} min → {batch_path}")

    # ====================== MERGE ======================
    all_batches = sorted(glob.glob(os.path.join(out_dir, "batch_*.csv")))
    if len(all_batches) < n_batches:
        print(f"⚠ Only {len(all_batches)}/{n_batches} batches completed.")
        return None

    print("\nAll batches done. Merging...")
    df = pd.concat([pd.read_csv(f) for f in all_batches], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df.to_csv(final_path, index=False)

    print(f"\n✅ Final dataset saved: {final_path}")
    print(f"Shape: {df.shape}")
    print("\nPDE distribution:\n", df["pde_type"].value_counts())
    print(f"Ill-conditioned rate: {df['ill_conditioned'].mean():.2%}")

    ill_cond_by_pde = (
        df.groupby("pde_type")["ill_conditioned"]
          .agg(count="size", ill_cond_rate="mean")
          .sort_values("ill_cond_rate", ascending=False)
    )
    ill_cond_by_pde["ill_cond_rate"] = (ill_cond_by_pde["ill_cond_rate"] * 100).round(2)
    ill_cond_by_pde = ill_cond_by_pde.rename(columns={"ill_cond_rate": "ill_cond_pct"})

    print("\nIll-conditioned % by PDE type:\n")
    print(ill_cond_by_pde.to_string())

    return df


# ====================== RUN ======================
if __name__ == "__main__":
    df = generate_dataset_chunked(
        n_samples=8000,
        batch_size=300,
        out_dir="/content/drive/MyDrive/matrix_batches",
        final_path="matrix_dataset_z.csv",
        seed=42
    )
if df is not None:
    try:
        from google.colab import files
        files.download("matrix_dataset_z.csv")
    except:
        print("File is saved. You can download it from the Files panel.")

Mounted at /content/drive
Found 9 completed batches. Resuming...
Batch  10/27 | size=300 | skipped=0 | time=55.3 min → /content/drive/MyDrive/matrix_batches/batch_0009.csv
Batch  11/27 | size=300 | skipped=0 | time=93.4 min → /content/drive/MyDrive/matrix_batches/batch_0010.csv
Batch  12/27 | size=300 | skipped=0 | time=129.7 min → /content/drive/MyDrive/matrix_batches/batch_0011.csv
Batch  13/27 | size=300 | skipped=0 | time=151.0 min → /content/drive/MyDrive/matrix_batches/batch_0012.csv
Batch  14/27 | size=300 | skipped=0 | time=188.0 min → /content/drive/MyDrive/matrix_batches/batch_0013.csv
Batch  15/27 | size=300 | skipped=0 | time=231.5 min → /content/drive/MyDrive/matrix_batches/batch_0014.csv
Batch  16/27 | size=300 | skipped=0 | time=285.9 min → /content/drive/MyDrive/matrix_batches/batch_0015.csv
Batch  17/27 | size=300 | skipped=0 | time=316.8 min → /content/drive/MyDrive/matrix_batches/batch_0016.csv
Batch  18/27 | size=300 | skipped=0 | time=322.0 min → /content/drive/MyD

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# WAVE IMPLICIT CHANGED!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import glob
import time
import numpy as np
import pandas as pd
from scipy.sparse import diags, kron, eye, csr_matrix, bmat, random as sp_random
from scipy.sparse.linalg import gmres, splu
import warnings

warnings.filterwarnings("ignore")
from IPython.display import HTML, Javascript
HTML('''<script>
function ClickConnect(){console.log("Keep alive"); document.querySelector("colab-connect-button").click()}
setInterval(ClickConnect, 60000);
</script>''')

# ====================== TUNABLE ILL-CONDITIONING KNOBS ======================
HELMHOLTZ_SINGULAR_PROB = 0.30
WAVE_EXTREME_PROB       = 0.22

# ====================== HELPER FUNCTIONS ======================

def build_1d_base(n, bc="dirichlet", rng=None):
    if rng is None:
        rng = np.random.default_rng()
    main = 2.0 * np.ones(n)
    off  = -1.0 * np.ones(n - 1)
    T    = diags([main, off, off], [0, -1, 1], format="lil")
    if bc == "neumann":
        T[0, 0] = T[n-1, n-1] = 1.0
    elif bc == "robin":
        alpha = rng.uniform(0.5, 5.0)
        T[0, 0] = T[n-1, n-1] = 2.0 + alpha
    return csr_matrix(T)

def apply_symmetric_stretching(A, N, rng):
    s = np.exp(rng.uniform(-1.4, 1.4, size=N))
    sqrtD = diags(np.sqrt(s), format="csr")
    return sqrtD @ A @ sqrtD

def apply_grid_stretching(A, N, rng):
    s = np.exp(rng.uniform(-1.4, 1.4, size=N))
    return diags(s, format="csr") @ A

def apply_sparsity_noise(A, N, rng, density=0.0015, symmetric=False):
    fro       = np.sqrt(np.sum(A.data**2) + 1e-12)
    noise_mag = 5e-4 * fro / (N + 1e-12)
    E         = sp_random(N, N, density=density, format="csr",
                          random_state=int(rng.integers(0, 2**31-1)))
    E.data   *= noise_mag
    if symmetric:
        E = (E + E.T) * 0.5
    return A + E

def robust_smax(A, rng, n_iter=25):
    try:
        N = A.shape[0]
        x = rng.standard_normal(N)
        x /= np.linalg.norm(x) + 1e-12
        for _ in range(n_iter):
            x   = A.T @ (A @ x)
            nrm = np.linalg.norm(x)
            if nrm < 1e-12: break
            x  /= nrm
        smax = np.sqrt(max(np.dot(x, A.T @ (A @ x)), 0.0))
        return max(smax, 1e-6)
    except:
        return np.sqrt(np.sum(A.data**2) + 1e-12)

def robust_smin(A, reg_shift=0.0, n_iter=15):
    try:
        N = A.shape[0]
        if not np.isfinite(A.data).all():
            return None
        fro = np.sqrt(np.sum(A.data**2))
        if fro > 1e30 or fro < 1e-30:
            scale       = fro
            A_scaled    = A * (1.0 / scale)
            smin_scaled = robust_smin(A_scaled, reg_shift / scale, n_iter)
            return smin_scaled * scale if smin_scaled is not None else None
        if reg_shift > 0:
            M = A.T @ A + (reg_shift**2) * eye(N, format="csr")
        else:
            M = A.T @ A
        lu = splu(M.tocsc(), permc_spec='COLAMD')
        x  = np.ones(N, dtype=np.float64) / np.sqrt(N)
        for _ in range(n_iter):
            x   = lu.solve(x)
            nrm = np.linalg.norm(x)
            if nrm < 1e-12: return None
            x  /= nrm
        rq        = np.dot(x, M @ x)
        shift_sq  = reg_shift ** 2 if reg_shift > 0 else 0.0
        eigenval  = max(rq - shift_sq, 1e-30)
        smin      = np.sqrt(eigenval)
        return max(smin, 1e-15)
    except:
        return None

def cond_estimate_lu(A, already_normalized=False, rng=None):
    try:
        N = A.shape[0]
        if not np.isfinite(A.data).all():
            return None
        if not already_normalized:
            fro = np.sqrt(np.sum(A.data**2))
            if fro < 1e-30 or fro > 1e30:
                return None
            A_scaled = A * (1.0 / fro)
        else:
            A_scaled = A
        norm_A   = float(np.max(np.abs(A_scaled).sum(axis=0)))
        lu       = splu(A_scaled.tocsc(), permc_spec='COLAMD')
        rng_p    = rng if rng is not None else np.random.default_rng()
        est      = 0.0
        for _ in range(6):
            e   = rng_p.choice([-1.0, 1.0], size=N)
            x   = lu.solve(e)
            est = max(est, np.linalg.norm(x, 1) / (np.linalg.norm(e, 1) + 1e-12))
        cond   = norm_A * est
        return float(np.log10(max(cond, 1.0) + 1e-12))
    except:
        return None

def gmres_decay_feature(A, N):
    b         = np.ones(N, dtype=np.float64)
    residuals = []
    try:
        def cb(r):
            residuals.append(float(np.linalg.norm(r)) if hasattr(r, '__len__') else float(r))
        gmres(A, b, maxiter=8, callback=cb, callback_type='pr_norm', atol=1e-8)
    except:
        pass
    if len(residuals) > 1:
        return float(np.clip(residuals[-1] / (residuals[0] + 1e-12), 0.0, 10.0))
    return 1.0

# ====================== FEATURE HELPERS ======================
def bandwidth_estimate(A):
    rows, cols = A.nonzero()
    return float(np.max(np.abs(rows - cols))) if len(rows) > 0 else 0.0

def sparsity_entropy(A):
    nnz_per_row = A.getnnz(axis=1)
    p = nnz_per_row.astype(float) / (nnz_per_row.sum() + 1e-12)
    p = p[p > 0]
    return -np.sum(p * np.log2(p + 1e-12)) if len(p) > 0 else 0.0

def off_diag_decay(A, rng, n_samples=100):
    N = A.shape[0]
    indptr, indices, data = A.indptr, A.indices, A.data
    rates = []
    n_draw = min(n_samples, N)
    idxs = rng.integers(0, N, size=n_draw)
    for i in idxs:
        start, end = indptr[i], indptr[i + 1]
        nnz = end - start
        if nnz > 3:
            row_idx = indices[start:end]
            row_val = data[start:end]
            dists      = np.abs(row_idx - i)
            vals       = np.abs(row_val)
            sorted_idx = dists.argsort()
            rates.append(np.mean(vals[sorted_idx][1:5]) /
                         (vals[sorted_idx[0]] + 1e-12))
    raw = np.mean(rates) if rates else 1.0
    return float(np.log1p(raw))

def gershgorin_features(diag, row_abs_sums):
    radii = row_abs_sums - np.abs(diag)
    return {
        "gersh_radius_max":         float(radii.max()),
        "gersh_radius_mean":        float(radii.mean()),
        "gersh_radius_std":         float(radii.std()),
        "gersh_origin_overlap_pct": float(np.mean(radii >= np.abs(diag))),
        "gersh_min_gap":            float(np.min(np.abs(diag) - radii)),
    }

def spectral_features(A, rng, diag, row_abs_sums, k=3):
    N      = A.shape[0]
    result = {}
    try:
        Q      = rng.standard_normal((N, k + 1))
        Q, _   = np.linalg.qr(Q)
        M_small = Q.T @ (A.T @ (A @ Q))
        eigs   = np.sort(np.abs(np.linalg.eigvalsh(M_small)))[::-1]
        for i, e in enumerate(eigs[:k]):
            result[f"rayleigh_eig{i}"] = float(e)
    except:
        for i in range(k):
            result[f"rayleigh_eig{i}"] = 0.0
    try:
        diag_abs    = np.abs(diag)
        row_offdiag = row_abs_sums - diag_abs
        dominance   = diag_abs - row_offdiag
        result["dom_q10"] = float(np.percentile(dominance, 10))
        result["dom_q50"] = float(np.percentile(dominance, 50))
        result["dom_q90"] = float(np.percentile(dominance, 90))
    except:
        result["dom_q10"] = result["dom_q50"] = result["dom_q90"] = 0.0
    return result

def row_col_statistics(A, diag, row_abs_sums):
    row_sums     = np.array(A.sum(axis=1)).flatten()
    col_sums     = np.array(A.sum(axis=0)).flatten()
    off_diag_max = row_abs_sums - np.abs(diag)
    return {
        "row_sum_min":         float(row_sums.min()),
        "row_sum_max":         float(row_sums.max()),
        "row_sum_std":         float(row_sums.std()),
        "col_sum_min":         float(col_sums.min()),
        "col_sum_max":         float(col_sums.max()),
        "col_sum_std":         float(col_sums.std()),
        "min_diag_dominance":  float(np.min(np.abs(diag) - off_diag_max)),
        "avg_diag_dominance":  float(np.mean(np.abs(diag) - off_diag_max)),
        "diag_dominant_ratio": float(np.mean(np.abs(diag) > off_diag_max)),
    }
# ====================== PDE BUILDERS ======================

def build_poisson(n, N, h, bc, rng):
    I    = eye(n, format="csr")
    Tx   = build_1d_base(n, bc, rng)
    Ty   = build_1d_base(n, bc, rng)
    cx   = 10 ** rng.uniform(-4.5, 6.0)
    cy   = 10 ** rng.uniform(-4.5, 6.0)
    jump = 10 ** rng.uniform(0.4, 8.0)
    L    = (cx * kron(I, Tx) + cy * kron(Ty, I)) / h**2
    coeff = np.ones(N); coeff[N//2:] *= jump
    sqrtD = diags(np.sqrt(coeff), format="csr")
    return csr_matrix(sqrtD @ L @ sqrtD, dtype=np.float64), 0.0, "poisson"

def build_convdiff(n, N, h, bc, rng):
    I    = eye(n, format="csr")
    Tx   = build_1d_base(n, bc, rng)
    Ty   = build_1d_base(n, bc, rng)
    ex   = 10 ** rng.uniform(-12.0, -1.5)
    ey   = 10 ** rng.uniform(-12.0, -1.5)
    v    = 10 ** rng.uniform(-1.0, 4.0)
    jump = 10 ** rng.uniform(2.0, 9.0)
    L    = (ex * kron(I, Tx) + ey * kron(Ty, I)) / h**2
    coeff = np.ones(N); coeff[N//2:] *= jump
    D    = diags(coeff, format="csr")
    e = np.ones(N)
    Adv = (v / h) * diags([e, -e[:-1]], [0, -1], shape=(N, N), format="csr")
    return csr_matrix(D @ L + D @ Adv, dtype=np.float64), 0.0, "convection_diffusion"

def build_helmholtz(n, N, h, bc, rng):

    I   = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L   = (kron(I, T) + kron(T, I)) / h**2
    eigs_1d = np.sort(np.linalg.eigvalsh(T.toarray())) / h**2
    eigs_1d = np.clip(eigs_1d, 0.0, None)

    lam_min = 2.0 * eigs_1d[0]
    lam_max = 2.0 * eigs_1d[-1]

    i_idx = int(rng.integers(0, n))
    j_idx = int(rng.integers(0, n))
    lam_target = eigs_1d[i_idx] + eigs_1d[j_idx]
    lam_target = max(lam_target, 1e-10)

    if rng.random() < HELMHOLTZ_SINGULAR_PROB:
        target_log10 = rng.uniform(5.5, 13.0)
        gap = lam_target / (10 ** target_log10)
        sign = rng.choice([-1.0, 1.0])
        k_sq = lam_target - sign * gap * rng.uniform(0.9, 1.3)
    else:
        if rng.random() < 0.65:
            k_sq = -lam_max * rng.uniform(3.0, 30.0)
        else:
            k_sq = lam_min * rng.uniform(0.4, 1.2)

    k_sq = float(np.clip(k_sq, -400.0 * lam_max, lam_max * 0.9995))

    if abs(lam_target - k_sq) < 1e-12 * max(lam_target, 1.0):
        k_sq -= max(lam_target, 1.0) * 3e-8

    A = csr_matrix(L - k_sq * eye(N, format="csr"), dtype=np.float64)
    return A, 0.0, "helmholtz"

def build_advreact(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)
    if rng.random() < 0.70:
        eps   = 10 ** rng.uniform(-12.0, -4.0)
        sigma = 10 ** rng.uniform(5.0, 11.0)
        jump  = 10 ** rng.uniform(5.0, 14.0)
    else:
        eps   = 10 ** rng.uniform(-9.0, -2.0)
        sigma = 10 ** rng.uniform(-3.5, 4.5)
        jump  = 10 ** rng.uniform(3.0, 12.0)
    L     = (eps * kron(I, Tx) + eps * kron(Ty, I)) / h**2
    coeff = np.full(N, sigma); coeff[N//2:] = sigma / jump
    return csr_matrix(L + diags(coeff, format="csr"),
                      dtype=np.float64), 0.0, "advection_reaction"

def build_biharmonic(n, N, h, bc, rng):
    I     = eye(n, format="csr")
    T     = build_1d_base(n, bc, rng)
    L     = (kron(I, T) + kron(T, I)) / h**2
    scale = 10 ** rng.uniform(0.0, 3.5)
    sqrtD = diags(np.sqrt(scale) * np.ones(N), format="csr")
    Ls    = sqrtD @ L @ sqrtD
    return csr_matrix(Ls @ Ls, dtype=np.float64), 0.0, "biharmonic"

def build_stokes_saddle(n, N, h, bc, rng):

    I_n = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L2d = (kron(I_n, T) + kron(T, I_n)) / h**2

    e  = np.ones(N)
    Bx = (1.0 / h) * diags([-e, e[:-1]], [0, 1], shape=(N, N), format="csr")
    By = (1.0 / h) * diags([e, -e[:N - n]], [0, n],  shape=(N, N), format="csr")

    B_scale = 1.0 / h

    if rng.random() < 0.45:
        nu    = 10 ** rng.uniform(0.5, 2.5)
        eps_p = 10 ** rng.uniform(-3.0, 0.0)
    else:
        nu    = 10 ** rng.uniform(-5.0, -2.0)
        eps_p = 10 ** rng.uniform(-12.0, -6.0)

    A = bmat([[nu * L2d,           csr_matrix((N, N)), Bx.T],
              [csr_matrix((N, N)), nu * L2d,           By.T],
              [Bx,                 By, -eps_p * eye(N, format="csr")]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "stokes_saddle"


def build_tensor_diffusion(n, N, h, bc, rng):
    I      = eye(n, format="csr")
    T      = build_1d_base(n, bc, rng)
    max_log = rng.uniform(3.5, 9.0)
    log_a1  = np.linspace(0, max_log, N);                  rng.shuffle(log_a1)
    log_a2  = np.linspace(0, max_log * rng.uniform(0.4, 0.9), N); rng.shuffle(log_a2)
    Kx = diags(10**log_a1, format="csr")
    Ky = diags(10**log_a2, format="csr")
    A  = (Kx @ kron(I, T) + Ky @ kron(T, I)) / h**2
    if rng.random() < 0.5:
        theta = rng.uniform(0, np.pi / 3)
        cross = 10 ** rng.uniform(-2.5, 1.0)
        off   = diags([np.cos(theta) * np.sin(theta) * cross * np.ones(N - 1)] * 2,
                      [-1, 1], format="csr")
        A = A + off
    if rng.random() < 0.45:
        A = A * 10 ** rng.uniform(-1.5, 3.0)
    return csr_matrix(A, dtype=np.float64), 0.0, "tensor_diffusion"

def build_parabolic(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    T  = build_1d_base(n, bc, rng)
    K  = (kron(I, T) + kron(T, I)) / h**2
    dt = 10 ** rng.uniform(-8.0, 5.0)
    M  = diags(10 ** rng.uniform(-8.0, 8.0, size=N), format="csr")
    A  = M + dt * K
    if rng.random() < 0.75: A = A * 10 ** rng.uniform(-4.0, 8.0)
    if rng.random() < 0.55: A = A * 10 ** rng.uniform(4.0, 10.0)
    return csr_matrix(A, dtype=np.float64), 0.0, "parabolic"

def build_hyperbolic(n, N, h, bc, rng):

    vx  = 10 ** rng.uniform(-1.0, 7.0) * rng.choice([-1, 1])
    vy  = 10 ** rng.uniform(-1.0, 7.0) * rng.choice([-1, 1])
    eps = 10 ** rng.uniform(-8.0, 1.5)

    e = np.ones(N)
    if vx >= 0:
        Adv_x = (abs(vx) / h) * diags([e, -e[:-1]],  [0, -1], shape=(N, N), format="csr")
    else:
        Adv_x = (abs(vx) / h) * diags([-e, e[:-1]], [0,  1], shape=(N, N), format="csr")
    if vy >= 0:
        Adv_y = (abs(vy) / h) * diags([np.ones(N), -np.ones(N - n)],
                                       [0, -n], shape=(N, N), format="csr")
    else:
        Adv_y = (abs(vy) / h) * diags([-np.ones(N), np.ones(N - n)],
                                       [0,  n], shape=(N, N), format="csr")

    if rng.random() < 0.50:
        log_scale = rng.uniform(0.0, 10.0)
        d_vals    = np.ones(N)
        idx       = rng.choice(N, size=N // 2, replace=False)
        d_vals[idx] = 10 ** log_scale
        D   = diags(d_vals, format="csr")
        A   = D @ (Adv_x + Adv_y) + eps * D
    else:
        A = Adv_x + Adv_y + eps * eye(N, format="csr")

    return csr_matrix(A, dtype=np.float64), 0.0, "hyperbolic"

def build_anisotropic(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)

    if rng.random() < 0.50:
        log_base  = rng.uniform(-1.0, 1.0)
        log_ratio = rng.uniform(-1.5, 1.5)
    else:
        log_base  = rng.uniform(-1.0, 1.0)
        sign      = rng.choice([-1.0, 1.0])
        log_ratio = sign * rng.uniform(8.0, 14.0)

    ax = 10 ** log_base
    ay = 10 ** (log_base + log_ratio)
    ay = float(np.clip(ay, 1e-15, 1e15))   # clip after computation

    return csr_matrix((ax * kron(I, Tx) + ay * kron(Ty, I)) / h**2,
                      dtype=np.float64), 0.0, "anisotropic"

def build_mixed(n, N, h, bc, rng):

    if rng.random() < 0.48:

        I    = eye(n, format="csr")
        Tx   = build_1d_base(n, bc, rng)
        Ty   = build_1d_base(n, bc, rng)
        c    = 10 ** rng.uniform(-1.0, 2.0)
        L    = (c * kron(I, Tx) + c * kron(Ty, I)) / h**2
        A1   = csr_matrix(L, dtype=np.float64)
        A2   = csr_matrix(L * (10 ** rng.uniform(-0.5, 0.5)), dtype=np.float64)
        coupling = 10 ** rng.uniform(-8.0, -4.0)
    else:
        A1, _, _ = build_poisson(n, N, h, bc, rng)
        A2, _, _ = build_advreact(n, N, h, bc, rng)
        coupling = 10 ** rng.uniform(-1.0, 4.0)

    A = bmat([[A1, coupling * eye(N, format="csr")],
              [coupling * eye(N, format="csr"), A2]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "mixed"

def build_reaction_diffusion(n, N, h, bc, rng):
    I_n = eye(n, format="csr")
    T   = build_1d_base(n, bc, rng)
    L   = (kron(I_n, T) + kron(T, I_n)) / h**2
    if rng.random() < 0.82:
        a     = 10 ** rng.uniform(4.0, 10.0)
        delta = 10 ** rng.uniform(-8.0, -3.0)
        b     = a * (1.0 - delta)
        eps1  = 10 ** rng.uniform(-10.0, -4.0)
        eps2  = 10 ** rng.uniform(-10.0, -4.0)
        A11   =  eps1 * L - a * eye(N, format="csr")
        A22   =  eps2 * L - a * eye(N, format="csr")
        A12   = -b * eye(N, format="csr")
        A21   = -b * eye(N, format="csr")
    else:
        d1  = 10 ** rng.uniform(-3.0, 2.0)
        d2  = 10 ** rng.uniform(-3.0, 2.0)
        a11 = 10 ** rng.uniform(-2.0, 3.0) * rng.choice([-1, 1])
        a12 = 10 ** rng.uniform(-2.0, 2.0)
        a21 = 10 ** rng.uniform(-2.0, 2.0)
        a22 = 10 ** rng.uniform(-2.0, 3.0) * rng.choice([-1, 1])
        A11 = d1 * L - a11 * eye(N, format="csr")
        A12 = -a12 * eye(N, format="csr")
        A21 = -a21 * eye(N, format="csr")
        A22 = d2 * L - a22 * eye(N, format="csr")
    A = bmat([[A11, A12], [A21, A22]], format="csr")
    return csr_matrix(A, dtype=np.float64), 0.0, "reaction_diffusion"


def build_wave_implicit(n, N, h, bc, rng):
    I  = eye(n, format="csr")
    Tx = build_1d_base(n, bc, rng)
    Ty = build_1d_base(n, bc, rng)

    lam_max_L = (2.0 / h**2) * (1.0 - np.cos(np.pi * n / (n + 1)))

    het_mass_prob = 0.35
    m0 = 10 ** rng.uniform(-1.0, 1.0)
    if rng.random() < het_mass_prob:
        density_ratio = 10 ** rng.uniform(4.0, 12.0)
        m_vals = np.ones(N) * m0
        split = rng.integers(N // 4, 3 * N // 4)
        idx = rng.choice(N, size=split, replace=False)
        m_vals[idx] *= density_ratio
        M = diags(m_vals, format="csr")
    else:
        M = m0 * eye(N, format="csr")

    extreme = rng.random() < WAVE_EXTREME_PROB
    if rng.random() < 0.50:
        rho = 10 ** rng.uniform(7.0, 11.0) if extreme else 10 ** rng.uniform(-3.0, 1.5)
        cx  = 10 ** rng.uniform(-1.0, 2.0)
        cy  = cx * 10 ** rng.uniform(-0.5, 0.5)
        dt_sq = rho * m0 / (cx * lam_max_L + 1e-30)
        dt = float(np.sqrt(np.clip(dt_sq, 0, 1e20)))
    else:
        cx = 10 ** rng.uniform(1.0, 4.0)
        aniso_ratio = 10 ** rng.uniform(5.0, 9.0)
        cy = cx / aniso_ratio
        if rng.random() < 0.5:
            cx, cy = cy, cx
        rho = 10 ** rng.uniform(7.0, 11.0) if extreme else 10 ** rng.uniform(0.0, 4.0)
        dt_sq = rho * m0 / (max(cx, cy) * lam_max_L + 1e-30)
        dt = float(np.sqrt(np.clip(dt_sq, 0, 1e20)))

    K = (cx * kron(I, Tx) + cy * kron(Ty, I)) / h**2
    A = M + dt**2 * K
    A = csr_matrix(A, dtype=np.float64)
    if not np.isfinite(A.data).all():
        A.data = np.clip(A.data, -1e30, 1e30)

    return A, 0.0, "wave_implicit"
def build_heterogeneous_diffusion(n, N, h, bc, rng):
    I = eye(n, format="csr")
    T = build_1d_base(n, bc, rng)
    L = (kron(I, T) + kron(T, I)) / h**2
    k = np.ones(N)
    k[N//4:N//2] = 10 ** rng.uniform(0.5, 8.0)
    k[N//2:]     = 10 ** rng.uniform(0.8, 9.0)
    if rng.random() < 0.5: k *= 10 ** rng.uniform(-1.8, 1.8)
    sqrtD = diags(np.sqrt(k), format="csr")
    return csr_matrix(sqrtD @ L @ sqrtD, dtype=np.float64), 0.0, "heterogeneous_diffusion"

# ====================== BUILDERS DICT ======================
BUILDERS = {
    "poisson":                  build_poisson,
    "convection_diffusion":     build_convdiff,
    "helmholtz":                build_helmholtz,
    "advection_reaction":       build_advreact,
    "biharmonic":               build_biharmonic,
    "stokes_saddle":            build_stokes_saddle,
    "tensor_diffusion":         build_tensor_diffusion,
    "parabolic":                build_parabolic,
    "hyperbolic":               build_hyperbolic,
    "anisotropic":              build_anisotropic,
    "mixed":                    build_mixed,
    "reaction_diffusion":       build_reaction_diffusion,
    "wave_implicit":            build_wave_implicit,
    "heterogeneous_diffusion":  build_heterogeneous_diffusion,
}

# ====================== MAKE SAMPLE ======================
def make_sample(pde, grid_size, bc, rng):
    n      = int(grid_size)
    base_N = n * n
    h      = 1.0 / (n + 1)

    A, reg_shift, pde_name = BUILDERS[pde](n, base_N, h, bc, rng)
    N = A.shape[0]

    spd_families = {"poisson", "biharmonic", "parabolic",
                    "heterogeneous_diffusion", "tensor_diffusion"}
    is_spd = pde_name in spd_families

    if is_spd and rng.random() < 0.6:
        A = apply_symmetric_stretching(A, N, rng)
    elif rng.random() < 0.4:
        A = apply_grid_stretching(A, N, rng)

    if rng.random() < 0.25:
        A = apply_sparsity_noise(A, N, rng, symmetric=is_spd)

    fro = np.sqrt(np.sum(A.data ** 2))
    if fro < 1e-30 or not np.isfinite(fro):
        return None
    A_norm = A * (1.0 / fro)

    log10c = cond_estimate_lu(A_norm, already_normalized=True, rng=rng)

    if log10c is None:
        try:
            smax = robust_smax(A_norm, rng)
            smin = robust_smin(A_norm, reg_shift / fro)
            if smin is not None and smin > 1e-14:
                cond_num = smax / smin
                log10c   = float(np.log10(cond_num + 1e-12))
            else:
                return None
        except:
            return None

    cond_num = 10 ** log10c
    diag     = A_norm.diagonal()
    fro_norm = np.sqrt(np.sum(A_norm.data**2) + 1e-12)

    all_pde_names = list(BUILDERS.keys())
    abs_A_norm    = A_norm.copy()
    abs_A_norm.data = np.abs(abs_A_norm.data)
    row_abs_sums  = np.array(abs_A_norm.sum(axis=1)).flatten()
    col_abs_sums  = np.array(abs_A_norm.sum(axis=0)).flatten()
    matrix_1norm   = float(col_abs_sums.max())
    matrix_infnorm = float(row_abs_sums.max())
    diff = A_norm - A_norm.T
    symmetry_measure = float(np.sqrt(diff.multiply(diff).sum()) / fro_norm)

    features = {
        "diag_min":         float(diag.min()),
        "diag_max":         float(diag.max()),
        "diag_mean":        float(diag.mean()),
        "diag_std":         float(diag.std()),
        "diag_ratio":       float(np.min(np.abs(diag)) / (np.max(np.abs(diag)) + 1e-12)),
        "frobenius_norm":   float(fro_norm),
        "matrix_1norm":     matrix_1norm,
        "matrix_infnorm":   matrix_infnorm,
        "norm_ratio_1_inf": matrix_1norm / (matrix_infnorm + 1e-12),
        "symmetry_measure": symmetry_measure,
        "avg_nnz_row":      A_norm.nnz / N,
        "max_row_nnz":      int(A_norm.getnnz(axis=1).max()),
        "nnz_std":          float(np.std(A_norm.getnnz(axis=1))),
        "bandwidth":        bandwidth_estimate(A_norm),
        "bandwidth_ratio":  bandwidth_estimate(A_norm) / np.sqrt(N),
        "sparsity_entropy": sparsity_entropy(A_norm),
        "off_diag_decay":   off_diag_decay(A_norm, rng),
        **gershgorin_features(diag, row_abs_sums),
        **row_col_statistics(A_norm, diag, row_abs_sums),
        **spectral_features(A_norm, rng, diag, row_abs_sums),
        "gmres_decay":      gmres_decay_feature(A_norm, N),
        "mesh_size":        float(h),
        "grid_size":        int(grid_size),
        "block_size":       3 if pde_name == "stokes_saddle" else (
                            2 if pde_name in {"mixed", "reaction_diffusion"} else 1),
        "pde_type":         pde_name,
        "condition_number": float(cond_num),
        "log10_cond":       log10c,
        "ill_conditioned":  int(log10c >= 7.0),
    }
    for p in all_pde_names:
        features[f"is_{p}"] = int(pde == p)
    for b in ["dirichlet", "neumann", "robin"]:
        features[f"bc_{b}"] = int(bc == b)

    return features

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
n_wave_needed = 575
rng = np.random.default_rng(7)
grid_sizes = [20, 25, 30, 35, 40, 45, 50, 80]

new_wave_rows, i = [], 0
while i < n_wave_needed:
    gs = rng.choice(grid_sizes)
    bc = rng.choice(["dirichlet", "neumann", "robin"])
    s = make_sample("wave_implicit", gs, bc, rng)
    if s:
        new_wave_rows.append(s)
        i += 1

new_wave_df = pd.DataFrame(new_wave_rows)
print("new wave_implicit ill-cond rate:", new_wave_df["ill_conditioned"].mean())
df = pd.read_csv("matrix_dataset_z.csv")
merged = pd.concat([df[df["pde_type"] != "wave_implicit"], new_wave_df], ignore_index=True)
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)
merged.to_csv("matrix_dataset_z.csv", index=False)

print(merged.groupby("pde_type")["ill_conditioned"].mean().sort_values(ascending=False))

new wave_implicit ill-cond rate: 0.19304347826086957
pde_type
heterogeneous_diffusion    0.642735
mixed                      0.638614
advection_reaction         0.589474
tensor_diffusion           0.576364
poisson                    0.573883
convection_diffusion       0.569930
parabolic                  0.458404
reaction_diffusion         0.313589
anisotropic                0.292035
hyperbolic                 0.254025
biharmonic                 0.248214
helmholtz                  0.208481
wave_implicit              0.193043
stokes_saddle              0.168190
Name: ill_conditioned, dtype: float64
